# 03. Remaining Useful Life (RUL) Estimation Model
## Fan Predictive Maintenance System

This notebook trains a **Degradation-based RUL Regression Model** to estimate the number of days remaining until equipment failure (0 to 30 days scale).

In [ ]:
import sys
import os
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer
from src.anomaly_detection import AnomalyDetector
from src.rul_estimation import RULEstimator

print("Modules imported successfully.")

### 1. Data Ingestion and Feature Extraction

In [ ]:
preprocessor = DataPreprocessor(config_path='../config/config.yaml')
preprocessor.load_data('../data/predictive_maintenance_dataset.csv')
baseline = preprocessor.extract_baseline()

engineer = FeatureEngineer(config_path='../config/config.yaml')
baseline_eng, feature_names = engineer.engineer_features(baseline)
X_baseline = baseline_eng[feature_names].values
baseline_mean = X_baseline.mean(axis=0)

data_eng, _ = engineer.engineer_features(preprocessor.data)
X_all = data_eng[feature_names].values

# Load or train anomaly detector for anomaly score degradation proxy
detector = AnomalyDetector(config_path='../config/config.yaml')
detector.train(X_baseline)
anomaly_scores = detector.predict_proba(X_all)

print(f"Baseline mean vector computed for {len(feature_names)} features.")

### 2. Train RUL Estimator

In [ ]:
rul_model = RULEstimator(config_path='../config/config.yaml')
rul_model.train(X_all, baseline_mean, anomaly_scores)

# Predict RUL for all samples
rul_preds = rul_model.predict(X_all)

print(f"=== RUL Predictions (Days) ===")
print(f"Mean RUL: {rul_preds.mean():.2f} days")
print(f"Min RUL:  {rul_preds.min():.2f} days")
print(f"Max RUL:  {rul_preds.max():.2f} days")

### 3. RUL vs. Fault Condition Analysis

In [ ]:
df_eval = preprocessor.data.copy()
df_eval['Predicted_RUL'] = rul_preds
df_eval['Anomaly_Score'] = anomaly_scores

plt.figure(figsize=(12, 6))
sns.boxplot(data=df_eval, x='Condition', y='Predicted_RUL', palette='RdYlGn_r')
plt.title('Predicted Remaining Useful Life (RUL) by Fault Condition')
plt.ylabel('RUL (Days)')
plt.xticks(rotation=30)
plt.axhline(y=7.0, color='orange', linestyle='--', label='Warning Threshold (7 Days)')
plt.axhline(y=3.0, color='red', linestyle='--', label='Critical Threshold (3 Days)')
plt.legend()
plt.tight_layout()
plt.show()

### 4. Correlation between Anomaly Severity and RUL Countdown

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df_eval['Anomaly_Score'], df_eval['Predicted_RUL'], c=df_eval['Predicted_RUL'], cmap='viridis_r', alpha=0.6)
plt.colorbar(label='RUL (Days)')
plt.title('Anomaly Score vs. Remaining Useful Life')
plt.xlabel('Anomaly Probability Score (0 = Normal, 1 = Severe Fault)')
plt.ylabel('RUL (Days)')
plt.grid(True)
plt.show()

### 5. Save RUL Model

In [ ]:
os.makedirs('../models', exist_ok=True)
rul_model.save('../models/rul_estimator_model.pkl')
print("RUL model artifact saved to ../models/rul_estimator_model.pkl")